# Reading the Canada mortgage rate nowcast output

This notebook opens my dataset [Canada Mortgage Rate Nowcast Output](https://www.kaggle.com/datasets/johnnykhan1/canada-mortgage-rate-nowcast-output), which holds the saved models and the latest nowcast from my notebook [Canada Mortgage Rate Nowcaster](https://www.kaggle.com/code/johnnykhan1/canada-mortgage-rate-nowcaster).

It does three things: shows the latest nowcast, checks that the numbers in it add up, and compares each model's cross validated error with the simple baseline stored next to it.

In [ ]:
import glob, json, os
import pandas as pd

import matplotlib.pyplot as plt
BLUE, RED = "#011F5B", "#990000"
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "axes.edgecolor": BLUE,
                     "axes.labelcolor": BLUE, "xtick.color": BLUE, "ytick.color": BLUE, "text.color": BLUE,
                     "axes.titleweight": "bold", "axes.spines.top": False, "axes.spines.right": False})

root = os.path.dirname(glob.glob("/kaggle/input/**/nowcast_latest.csv", recursive=True)[0])
for dirpath, _, files in os.walk(root):
    for f in sorted(files):
        p = os.path.join(dirpath, f)
        print(os.path.relpath(p, root), os.path.getsize(p), "bytes")

## The latest nowcast

In [ ]:
now = pd.read_csv(os.path.join(root, "nowcast_latest.csv"))
now

## Check: does each nowcast equal the last published rate plus the predicted change?

The file rounds to two decimals, so I allow a gap of up to 0.01.

In [ ]:
gap = (now.last_published + now.predicted_change - now.nowcast_rate).abs()
now.assign(gap=gap.round(4))
print("All rows add up:", bool((gap <= 0.01).all()))

## Model info for each variation

In [ ]:
info = {}
for p in sorted(glob.glob(os.path.join(root, "*", "model_info.json"))):
    info[os.path.basename(os.path.dirname(p))] = json.load(open(p))
for name, d in info.items():
    print(name, "| target:", d["target"], "| trained through:", d["trained_through"], "| scikit learn", d["sklearn_version"])
    print("  features:", ", ".join(d["features"]))
rows = []
for name, d in info.items():
    for h in d["horizons"]:
        rows.append({"variation": name, "horizon_months": h,
                     "cv_mae": d["cv_mae"][str(h)], "baseline_cv_mae": d["baseline_cv_mae"][str(h)]})
mae = pd.DataFrame(rows)
mae["error_cut_vs_baseline_%"] = (100 * (1 - mae.cv_mae / mae.baseline_cv_mae)).round(1)
mae

## Cross validated error against the baseline

Lower is better. Units are percentage points of the target rate.

In [ ]:
import numpy as np
names = list(info)
fig, axes = plt.subplots(1, len(names), figsize=(5.5 * len(names), 4), sharey=True)
axes = np.atleast_1d(axes)
for ax, name in zip(axes, names):
    m = mae[mae.variation == name]
    x = np.arange(len(m))
    ax.bar(x - 0.2, m.baseline_cv_mae, 0.4, color=RED, label="baseline")
    ax.bar(x + 0.2, m.cv_mae, 0.4, color=BLUE, label="model")
    ax.set_xticks(x, [f"{h} month" for h in m.horizon_months])
    ax.set_title(name)
    ax.set_ylabel("Cross validated MAE (pp)")
axes[0].legend(frameon=False)
plt.tight_layout()
plt.show()

## Can the saved models be opened here?

They were saved with scikit learn 1.6.1. If the version in this notebook differs, loading may warn or fail, so this cell reports what happens instead of hiding it.

In [ ]:
import sklearn, joblib, warnings
print("scikit learn here:", sklearn.__version__)
for p in sorted(glob.glob(os.path.join(root, "*", "*.joblib"))):
    try:
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")
            m = joblib.load(p)
        print(os.path.relpath(p, root), "loaded:", type(m).__name__, "| warnings:", len(w))
    except Exception as e:
        print(os.path.relpath(p, root), "did not load:", type(e).__name__, str(e)[:120])

## Notes

This is a model output snapshot, trained through 2026-07-01 on Bank of Canada data. It is a statistical estimate, not advice, and it is not refreshed on a schedule.